In [31]:
# Chap 16. 다양한 댓글 모으기
# 뉴스 기사의 댓글 모으기 - 미세먼지 / 스모그
# 테스트 기사 URL :
# https://news.naver.com/main/read.nhn?mode=LSD&mid=shm&sid1=102&oid=056&aid=0010661268

#Step 1. 필요한 모듈과 라이브러리를 로딩합니다.

from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
import time
import math
import numpy
import pandas as pd
import random
import os
import re

#Step 2. 사용자에게 검색어 키워드를 입력 받고 저장할 폴더와 파일명을 설정합니다.
print("=" *80)
print(" Chap 16.휴넷 댓글 정보 수집하기")
print("=" *80)
print("\n")

query_txt = '휴넷댓글'
query_url = input('1.댓글을 크롤링할 휴넷의 URL을 입력하세요: ')
cnt = int(input('2.크롤링 할 건수는 몇건입니까?(10건단위로 입력): '))
page_cnt = math.ceil(cnt / 10)

f_dir = input("3.파일을 저장할 폴더명만 쓰세요(예:c:\\py_temp\\):")
if f_dir=='' :
    f_dir='c:\\py_temp\\'

if cnt > 10 :
      print("요청 건수가 많아서 시간이 제법 소요되오니 잠시만 기다려 주세요~~")
else :
      print("요청하신 데이터를 수집하고 있으니 잠시만 기다려 주세요~~")

# 저장될 파일위치와 이름을 지정합니다
n = time.localtime()
s = '%04d-%02d-%02d-%02d-%02d-%02d' % (n.tm_year, n.tm_mon, n.tm_mday, n.tm_hour, n.tm_min, n.tm_sec)

os.makedirs(f_dir+s+'-'+query_txt)
os.chdir(f_dir+s+'-'+query_txt)

ff_name=f_dir+s+'-'+query_txt+'\\'+s+'-'+query_txt+'.txt'
fc_name=f_dir+s+'-'+query_txt+'\\'+s+'-'+query_txt+'.csv'
fx_name=f_dir+s+'-'+query_txt+'\\'+s+'-'+query_txt+'.xls'

#Step 3. 크롬 드라이버를 사용해서 웹 브라우저를 실행합니다.

s_time = time.time( )

s = Service("c:/py_temp/chromedriver.exe")
driver = webdriver.Chrome(service=s)

driver.get(query_url)
driver.maximize_window()
time.sleep(5)

#Step 4. 현재 총 리뷰 건수를 확인하여 사용자의 요청건수와 비교 후 동기화합니다
html = driver.page_source
soup = BeautifulSoup(html, 'html.parser')

result= soup.find('div','count reviewInfo').find('strong','reviewCnt')		#결과 뽑는거. 이게 중요!*!*!*
result2 = result.get_text()		#리뷰 개수 뽑는 것

print("=" *80)
result3 = result2.replace(",","")		#1,000건이 넘으면 중간에 ,가 있어서 제거하는거.
result4 = re.search("\d+",result3)
search_cnt = int(result4.group())

if cnt > search_cnt :
    cnt = search_cnt

print("전체 검색 결과 건수 :",search_cnt,"건")
print("실제 최종 출력 건수",cnt)
print("실체 출력될 최종 페이지수" , page_cnt)


 Chap 16.휴넷 댓글 정보 수집하기




<>:72: SyntaxWarning: invalid escape sequence '\d'
<>:72: SyntaxWarning: invalid escape sequence '\d'
C:\Users\itwill\AppData\Local\Temp\ipykernel_12484\3735570498.py:72: SyntaxWarning: invalid escape sequence '\d'
  result4 = re.search("\d+",result3)


요청 건수가 많아서 시간이 제법 소요되오니 잠시만 기다려 주세요~~
전체 검색 결과 건수 : 27 건
실제 최종 출력 건수 27
실체 출력될 최종 페이지수 3


In [32]:
# Step 5. 사용자가 요청한 건수가 많을 경우 리뷰 더보기 버튼을 클릭합니다
# 최초 10건 수집후 댓글 더보기 버튼 클릭
# 아래 버튼을 눌러 첫 화면에 총 20건의 댓글이 나오게 만듦
driver.find_element(By.XPATH,'//*[@id="tab3"]/a').click()		#댓글 더보기(5->2
time.sleep(3)

#Step 6. 리뷰와 점수 등 내용 수집
writer_name2=[] # 리뷰 작성자 이름
writer_id2=[] # 리뷰 작성자 ID
review2=[] # 리뷰 내용
count = 0

for a in range(1,page_cnt+1) :

        print('이제 리뷰 정보를 수집합니다. 잠시만 기다려 주세요~~~~~~~~')
        
        #txt 파일에 저장하기 위해 파일 open하기
        f = open(ff_name, 'a',encoding='UTF-8')
        
        html = driver.page_source
        soup = BeautifulSoup(html, 'html.parser')
        
        reple_result = soup.find('div', id='reviewList').find('ul')
        slist = reple_result.find_all('li')
        
        for li in slist:
            count += 1
            print("\n")
            print("총 %s건 중 %s번째 댓글 수집 중입니다 =========" %(cnt,count))
            
            try :
                writer_name = li.find('div', class_='author').find_all('strong')[0].get_text()
            except AttributeError :
                writer_name='작성자 이름 없음'
                print("1.작성자 이름 :",writer_name)
            else :
                print("1.작성자 이름:",writer_name)
            f.write("\n")
            f.write("총 %s 건 중 %s 번째 리뷰 데이터를 수집합니다====" %(cnt,count) + "\n")
            f.write("1.작성자 이름:" + writer_name + "\n")
            writer_name2.append(writer_name)

            writer_id = li.find('div', class_='author').find_all('strong')[1].get_text()
            print("2.작성자ID:", writer_id)
            f.write("2.작성자이ID:"+writer_id + "\n")
            writer_id2.append(writer_id)

            try :
                li.find('div', class_='author').extract() 

                # 2. 이제 <li> 안에는 div가 통째로 사라지고 "예상보다..." 텍스트만 남게 됩니다.
                review = li.get_text(strip=True)
            except AttributeError :
                review='작성자에 의해 삭제된 댓글입니다'
                print("3.리뷰 :",review)
            else :
                print("3.리뷰:",review)
            f.write("3.리뷰:" + review + "\n")
            review2.append(review)
            
            time.sleep(0.2)
            
        if count == cnt :
            break
            
    # [수정된 부분] 바탕(div) 클릭 대신 명확한 다음 '페이지 번호' 클릭으로 변경
        if a < page_cnt:
            try:
                # 현재 페이지(a) 다음 번호(a+1)를 가진 링크 텍스트를 찾아 정확히 클릭
                driver.find_element(By.LINK_TEXT, str(a + 1)).click()
            except:
                    # 예외 발생 시 기존 방식 호환 유지 (숫자 클릭이 안 먹힐 경우)
                    driver.find_element(By.XPATH,'//*[@id="reviewList"]/div/div').click()   
            time.sleep(random.randrange(1,3)) # 3-8 초 사이에 랜덤으로 시간 선택
                
                

이제 리뷰 정보를 수집합니다. 잠시만 기다려 주세요~~~~~~~~


총 27건 중 1번째 댓글 수집 중입니다 =========
1.작성자 이름: 김*홍
2.작성자ID: nsr****
3.리뷰: 예상보다 상당히 괜찮은 교육이었음. 손코딩없이 CharGPT로 코딩한다는 점이 신기했는데, ChatGPT를 자유자재로 다루는 방법을 익힐 수 있음. Colab도 실습에서 매번 사용했는데, Colab을 사용하며 발생하는 다양한 오류를 수정하며, 특히 한글 데이터 다루는 법을 명확히 알 수 있음


총 27건 중 2번째 댓글 수집 중입니다 =========
1.작성자 이름: 윤*윤
2.작성자ID: kir****
3.리뷰: 저에게 딱 필요하던 수업이네요. 전 개발자가 아니라, 어떠한 패턴이 있고 패턴 분석을 찾아 시각화하는 스킬적인 면이 필요했는데, 요즘 트렌드에 딱 맞는 바이브코딩의 정석이네요! 파이썬 코드를 새로 배워야하나.. 고민이 많았는데 파이썬코드는 기본정도만 알아두고(사실 알필요도 없을것 같긴해요) 어떤 ai툴을 이용해서 분석기법만 잘 알고있다면 활용도가 무궁무진하다는걸 알았습니다. 실습 전 데이터에 이해를 함께 설명해주셔서 정말 유익했습니다! 다음과정 개설되면 또 듣고싶네요.


총 27건 중 3번째 댓글 수집 중입니다 =========
1.작성자 이름: 하*찬
2.작성자ID: nar****
3.리뷰: 학습관련 전반적 으로 만족 했으나 사내 수강시 VDI 내에서만 강의수강되고 실제 연습을 하면서 따라 수강하기 위해서는 Local pc 에서 프로그램을 설치하고 해야 하는데 지원이 되지않아 수강에 제약이 있어 힘들었습니다. 좀더 원할한 교육이 되도록 지원 되었으면 좋겠습니다.


총 27건 중 4번째 댓글 수집 중입니다 =========
1.작성자 이름: 정*진
2.작성자ID: dae****
3.리뷰: chat GPT 무료 버전으로는 해당 수업을 따라가기 어려웠음. 어느 순간 이해가 안되면 끝까지 이해하기 어려움


총 27건 중 5번째 댓글 수집 중입니다 ====

In [33]:
#Step 7. xls 형태와 csv 형태로 저장하기
news_reple = pd.DataFrame()
news_reple['작성자이름']=pd.Series(writer_name2)
news_reple['작성자ID']=pd.Series(writer_id2)
news_reple['리뷰내용']=pd.Series(review2)

# csv 형태로 저장하기
news_reple.to_csv(fc_name,encoding="utf-8-sig",index=True)

# 엑셀 형태로 저장하기
news_reple.to_excel(fx_name ,index=True , engine='openpyxl')

# Step 8. 요약 정보 출력하기
e_time = time.time( )
t_time = e_time - s_time

print("\n")
print("=" *120)
print("1.요청된 총 %s 건의 리뷰 중에서 실제 크롤링 된 리뷰수는 %s 건입니다" %(cnt,count))
print("2.총 소요시간은 %s 초 입니다 " %round(t_time,1))
print("3.파일 저장 완료: txt 파일명 : %s " %ff_name)
print("4.파일 저장 완료: csv 파일명 : %s " %fc_name)
print("5.파일 저장 완료: xls 파일명 : %s " %fx_name)
print("=" *120)
driver.close()



1.요청된 총 27 건의 리뷰 중에서 실제 크롤링 된 리뷰수는 27 건입니다
2.총 소요시간은 33.4 초 입니다 
3.파일 저장 완료: txt 파일명 : c:\py_temp\2026-03-11-15-50-52-휴넷댓글\2026-03-11-15-50-52-휴넷댓글.txt 
4.파일 저장 완료: csv 파일명 : c:\py_temp\2026-03-11-15-50-52-휴넷댓글\2026-03-11-15-50-52-휴넷댓글.csv 
5.파일 저장 완료: xls 파일명 : c:\py_temp\2026-03-11-15-50-52-휴넷댓글\2026-03-11-15-50-52-휴넷댓글.xls 
